# [강사용] 예제 05 LSTM 모델 사전 학습
### 수업 전에 강사가 한 번 실행합니다 · 교육생 배포용 아님

이 노트북은 예제 05에서 교육생이 불러올 **학습된 LSTM 모델 파일 2개**를 만듭니다.

| 만들어지는 파일 | 내용 |
|---|---|
| `lstm_deposit_rate.weights.h5` | 학습된 LSTM 가중치 |
| `lstm_config.json` | 입력 길이(12개월), 0~1 변환 기준값(최솟값·최댓값), 학습 기간 등 설정 |

- **런타임은 기본값(CPU)** 으로 실행합니다. GPU 런타임에서는 결과가 매번 조금씩 달라질 수 있습니다.
- 학습 기간은 **2016년 1월 ~ 2024년 12월**, 2025년 1월 ~ 2026년 6월(18개월)은 학습에 쓰지 않고 예제 05의 **시험 구간**으로 남겨 둡니다.
- 전체 실행 시간: Colab CPU 기준 1~2분.
- 배포한 파일로 예제 05를 실행하면 노트북에 적힌 오차 수치와 같은 결과가 나옵니다. 이 노트북으로 모델을 **다시** 만들면 수치가 조금 달라질 수 있으니, 예제 05 본문의 숫자도 함께 확인합니다.

In [ ]:
import os
import urllib.request
import json
import numpy as np
import pandas as pd
import plotly.express as px
import keras
import tensorflow as tf

DATA_URL = "https://raw.githubusercontent.com/jetsonai/AIFinancialDataAnalysis/main/Day1/data/"
FILE = "deposit_rate_monthly.csv"
if not os.path.exists(FILE):          # 파일이 없으면 GitHub에서 내려받습니다
    try:
        urllib.request.urlretrieve(DATA_URL + FILE, FILE)
    except Exception:                 # 인터넷이 막혔을 때만 직접 올리기
        from google.colab import files
        files.upload()
print("데이터 준비 완료:", os.path.exists(FILE), "/ Keras", keras.__version__)

## 1. 학습 설정

In [ ]:
설정 = {
    "window": 12,                 # 최근 12개월을 보고
    "units": 16,                  # LSTM 기억 칸 수
    "epochs": 400,                # 학습 반복 횟수
    "learning_rate": 0.005,
    "seed": 42,                   # 같은 결과를 다시 얻기 위한 고정값
    "train_end": "2024-12-01",    # 이 달까지를 학습에 사용
}

## 2. 데이터 준비 — 0~1 변환과 12개월 묶음

In [ ]:
df = pd.read_csv(FILE, parse_dates=["date"])
금리 = df["deposit_rate_pct"].to_numpy(dtype="float32")
월 = df["date"]

학습기간 = 월 <= 설정["train_end"]
설정["min"] = round(float(금리[학습기간].min()), 4)
설정["max"] = round(float(금리[학습기간].max()), 4)
축소 = (금리 - 설정["min"]) / (설정["max"] - 설정["min"])
print("0~1 변환 기준: 최솟값", 설정["min"], "/ 최댓값", 설정["max"])

In [ ]:
W = 설정["window"]
X = np.array([축소[i - W:i] for i in range(W, len(축소))])[..., None]   # 직전 12개월
y = 축소[W:]                                                           # 맞힐 달
대상 = np.arange(W, len(축소))                                          # 맞힐 달의 위치

학습 = (월.iloc[대상] <= 설정["train_end"]).to_numpy()
print("학습용 묶음:", 학습.sum(), "개 / 시험용 묶음:", (~학습).sum(), "개")

## 3. LSTM 학습

In [ ]:
keras.utils.set_random_seed(설정["seed"])
tf.config.experimental.enable_op_determinism()

model = keras.Sequential([
    keras.Input(shape=(W, 1)),
    keras.layers.LSTM(설정["units"]),
    keras.layers.Dense(1),
])
model.compile(optimizer=keras.optimizers.Adam(설정["learning_rate"]), loss="mse")
기록 = model.fit(X[학습], y[학습], epochs=설정["epochs"], batch_size=16, verbose=0)
print("학습 완료 · 마지막 손실:", round(기록.history["loss"][-1], 5))

In [ ]:
fig = px.line(y=기록.history["loss"], labels={"x": "반복(epoch)", "y": "손실(MSE)"},
              title="학습 손실 — 아래로 내려가며 평평해지면 학습이 된 것", log_y=True)
fig.show()

## 4. 시험 구간 확인 (2025년 1월 ~ 2026년 6월)

학습에 쓰지 않은 18개월로 LSTM과 기준모델(전월값 그대로)의 평균 오차를 비교합니다.

In [ ]:
예측 = model.predict(X[~학습], verbose=0).ravel() * (설정["max"] - 설정["min"]) + 설정["min"]
실제 = 금리[대상[~학습]]
전월값 = 금리[대상[~학습] - 1]
print("LSTM 평균 오차   :", round(float(np.abs(예측 - 실제).mean()), 3), "%p")
print("전월값 평균 오차 :", round(float(np.abs(전월값 - 실제).mean()), 3), "%p")

## 5. 모델 파일 저장 · 내려받기

두 파일을 교육생 배포 폴더에 `deposit_rate_monthly.csv`와 함께 넣습니다.

In [ ]:
model.save_weights("lstm_deposit_rate.weights.h5")
with open("lstm_config.json", "w", encoding="utf-8") as f:
    json.dump(설정, f, ensure_ascii=False, indent=2)
print(json.dumps(설정, ensure_ascii=False, indent=2))

In [ ]:
from google.colab import files
files.download("lstm_deposit_rate.weights.h5")
files.download("lstm_config.json")